<a href="https://colab.research.google.com/github/doney25/CaseStudy/blob/main/Data_Acquisition_Covid_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://disease.sh/v3/covid-19/vaccine/coverage/countries?lastdays=1

JSON → DataFrame<br>
 Remove duplicates<br>
Handle missing values<br>
Rename columns - vaccines to total_vaccines<br>
Convert datatypes if required<br>
 Sort by total vaccinationations<br>
Category - based on total vaccinations<br>
 new column - etl run time<br>
high vaccines category countries count<br>
Top 10<br>
Bottom 10<br>
summary with total countries , average vaccinations , max vaccinations , highest vaccinated country

In [30]:
import os
import time
from datetime import datetime
import numpy as np
import pandas as pd
import requests

In [31]:
def extract():
  url='https://disease.sh/v3/covid-19/vaccine/coverage/countries?lastdays=1'
  response=requests.get(url)
  Disease_data=response.json()
  Data=pd.DataFrame(Disease_data)
  Data["date"] = Data["timeline"].apply(lambda x: list(x.keys())[0])
  Data["Vaccines"] = Data["timeline"].apply(lambda x: list(x.values())[0])
  Data.drop(columns="timeline", inplace=True)
  Data_len=len(Data)
  print(f"Extracted {Data_len} rows")
  return Data


In [32]:
def transform(Data):
  Data.drop_duplicates(inplace=True)
  Data.dropna(inplace=True)
  Data.rename(columns={'Vaccines':'total_vaccines'},inplace=True)
  Data["date"]=pd.to_datetime(Data['date'])
  Data.sort_values(by='total_vaccines',ascending=False,inplace=True)
  Data.reset_index(drop=True, inplace=True)

  Data["Category"]=Data['total_vaccines'].apply(
      lambda x:'High Vaccine' if x > 4000000
      else 'Moderate Vaccine' if x > 500000
      else 'Mild vaccine')
  Data['run time']=datetime.now()
  high_vaccine_count=Data[Data['Category']=='High Vaccine'].shape[0]
  print(f"There are {high_vaccine_count} high vaccine countries.\n")

  return Data



In [33]:
def load(Data):
  print("Summary:\n")
  total_country=Data['country'].nunique()
  avg_vaccination=Data['total_vaccines'].mean()
  max_vaccination=Data['total_vaccines'].max()
  highest_vaccinated_country=Data.iloc[0]["country"]

  print(f"Total Countries: {total_country}")
  print(f"Average Vaccinations: {avg_vaccination:.2f}")
  print(f"Max Vaccinations: {max_vaccination}")
  print(f"Highest Vaccinated Country: {highest_vaccinated_country}\n")

  print(Data.head(10))
  print(Data.tail(10))

In [34]:
data=extract()
new_data=transform(data)
load(new_data)


Extracted 217 rows
There are 121 high vaccine countries.

Summary:

Total Countries: 217
Average Vaccinations: 62562559.41
Max Vaccinations: 3491077000
Highest Vaccinated Country: China

      country       date  total_vaccines      Category  \
0       China 2025-07-21      3491077000  High Vaccine   
1       India 2025-07-21      2206868000  High Vaccine   
2         USA 2025-07-21       676728782  High Vaccine   
3      Brazil 2025-07-21       486436436  High Vaccine   
4   Indonesia 2025-07-21       448199860  High Vaccine   
5       Japan 2025-07-21       383747738  High Vaccine   
6  Bangladesh 2025-07-21       361742554  High Vaccine   
7    Pakistan 2025-07-21       340974144  High Vaccine   
8     Vietnam 2025-07-21       266492149  High Vaccine   
9      Mexico 2025-07-21       223158993  High Vaccine   

                    run time  
0 2026-06-27 18:35:10.194103  
1 2026-06-27 18:35:10.194103  
2 2026-06-27 18:35:10.194103  
3 2026-06-27 18:35:10.194103  
4 2026-06-27 18:35: